In [1]:
from fits_io import scan_dataset, update_headers, dataset_metadata, object_in_fov
from fits_io import identify_objects, cleanup_intermediate_files 
from fits_io import flag_object_in_fov, load_dataset_objects
from reduccion import run_reduction, plot_reduction
from astrometria import run_astrometry
from stars_catalog import generate_refcat, plot_catalog_on_image
from combine import process_and_combine_images, plot_combined, remove_aligment_tempfiles
import pandas as pd
from pathlib import Path
from utils import load_config, Tee, load_target_coordinates
from datetime import datetime
import sys



# -----------------------------------------------------------------------------
# Load settings
# -----------------------------------------------------------------------------

config_file="config.yaml"
cfg = load_config(config_file)

BASE = Path(cfg["paths"]["BASE"])
data_path = Path(BASE, cfg["paths"]["data_dir"])
object_path = Path(data_path, cfg["paths"]["objects_dir"])
night = cfg["night"]
night_dir = Path(data_path, night)
objects_csv = Path(object_path, "objetos.csv")

gain = cfg["instrument"]["gain"]
rdnoise = cfg["instrument"]["rdnoise"]

steps = cfg["steps"]

print(f"Running pipeline in night dir: {night_dir}")
# -------------------------------------------------------------------------
# Redirect stdout/stderr to a single log file (append)
# -------------------------------------------------------------------------
log_file = night_dir / "pipeline.log"
#sys.stdout = Tee(log_file)
#sys.stderr = sys.stdout

print("\n"*2 + "="*80)
print(f"New pipeline run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


# -------------------------------------------------------------------------
# 1) Scan dataset
# -------------------------------------------------------------------------
dataset = scan_dataset(night_dir)

# -------------------------------------------------------------------------
# 2) Update headers
# -------------------------------------------------------------------------
if steps.get("update_headers", False):
    print("→ Updating FITS headers")
    update_headers(dataset["all"], gain=gain, rdnoise=rdnoise)

# -------------------------------------------------------------------------
# 3) Export metadata
# -------------------------------------------------------------------------
if steps.get("export_metadata", False):
    output_file = cfg["metadata"].get("dataset_metadata_file", "images_data.csv")
    print(f"→ Exporting metadata to {output_file}")
    metadata_file = dataset_metadata(dataset, night_dir, output_file=output_file,
                                         objects_csv=objects_csv)
    # Load objects list
    objects = load_dataset_objects(night_dir, output_file)
    
    # Create a folder in objects dir for each object
    for objname in objects:
        obj_dir = Path(object_path, objname)
        obj_dir.mkdir(parents=True, exist_ok=True)


# -------------------------------------------------------------------------
# 4) Reduction
# -------------------------------------------------------------------------
if steps.get("reduction", False):
    print("→ Running reduction")

    reduction_cfg = cfg.get("reduction", {})
    reduction_result = run_reduction(
        dataset=dataset,
        zero_correction=reduction_cfg.get("zero_correction", True),
        flat_correction=reduction_cfg.get("flat_correction", True),
        dark_correction=reduction_cfg.get("dark_correction", False),
    )
else:
    reduction_result = None
    
# -------------------------------------------------------------------------
# 4.a) Plot de reducción
# -------------------------------------------------------------------------
if cfg["qc"].get("reduction_images", False):
    print("→ [QC] Generating reduction plots")
    for objname in objects:
        print(f"   → Object: {objname}")

        plot_reduction(
            dataset=output_file,
            night_dir=night_dir,
            objname=objname,
            output_name=f"reduction_{objname}.png",
            show=False,
            overwrite=True
        )
    print("✓ [QC] Reduction plots generated")

# -------------------------------------------------------------------------
# 5) Astrometry
# -------------------------------------------------------------------------
if steps.get("astrometry", False):
    print("→ Running astrometry")

    astro_cfg = cfg.get("astrometry", {})
    run_astrometry(
        dataset=dataset,
        output_dir=night_dir,
        api_key=astro_cfg["api_key"],
        overwrite=astro_cfg.get("overwrite", True),
    )
    dataset = scan_dataset(night_dir)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=output_file)
# -------------------------------------------------------------------------
# 5.5) Images contains its object?
# -------------------------------------------------------------------------
if steps.get("contains_obj", False):
    print("→ Flagging not contained images")
    df = flag_object_in_fov(metadata_file,
                       objects_csv,
                       night_dir,
                       overwrite=False)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=output_file)

# -------------------------------------------------------------------------
# 6) Combination
# -------------------------------------------------------------------------
if steps.get("combine", False):
    print("→ Combining images per object and filter")
    
    filters = cfg.get("filters", ["I", "V"])
    for objname in objects:
        print(f"   → Object: {objname}")
        print(f"      → Generating reference catalog")
        # Load metadata for the object
        ra, dec = load_target_coordinates(objname, objects_csv)
        alig_cfg = cfg.get("aligment", {})["catalog"]
        # Load image path for the object
        ds = pd.read_csv(Path(night_dir, output_file))
        img_files = ds[(ds["OBJECT"]==objname)&(ds["ASTROMET"]=="yes")]["FILENAME"].values
        img_path = night_dir / img_files[0]

        alig_cat_path = generate_refcat(
            objname=objname, ra_center=ra, dec_center=dec,
            img_path=img_path, objects_dir=object_path,
            fov_frac=alig_cfg.get("fov_frac", 0.3),
            min_mag=alig_cfg.get("min_mag", 9),
            max_mag=alig_cfg.get("max_mag", 12),
            use_catalogs=alig_cfg.get("use_catalogs", "all"),
            plot=alig_cfg.get("plot", False),
            overwrite=alig_cfg.get("overwrite", False),
        )

        for filt in filters:
            print(f"      → Filter: {filt}")
            images_to_combine = [night_dir / im for im in img_files if im[-14].upper() == filt.upper()]
            if len(images_to_combine) == 0:
                print(f"         ! No images found for filter {filt}. Skipping.")
                continue
            print(f"         ✓ Found {len(images_to_combine)} images for filter {filt}.")
            combined_im = process_and_combine_images(images_to_combine, objname,  filt, object_path, night_dir, objects_csv)
            # Plot aligment reference catalog on combined images (unnecessary step)
            plot_catalog_on_image(fits_file=combined_im,
                                  catalog=alig_cat_path,
                                  obj_ra=ra,
                                  obj_dec=dec,
                                  out_png=Path(night_dir, f"{objname}{filt}_comb_alig_cat.png"),
                                  title=f"{objname} – Aligment catalog")
            if cfg["combine"].get("remove_temp_files", False):
                print("      → Removing temporary files")
                removed = remove_aligment_tempfiles(filt, night_dir)
    dataset = scan_dataset(night_dir)
    metadata_file = dataset_metadata(dataset, night_dir,
                                    output_file=output_file)

if cfg["qc"].get("combined_images", False):
    print("→ Generating combined plots")
    for objname in objects:
        print(f"   → Object: {objname}")
        image_files = [file for file in dataset["images_combined"] if objname in file.name]
        if len(image_files) == 0:
            print(f"      ! No combined images found for object {objname}. Skipping.")
            continue  
        plot_combined(objname, image_files,  night_dir, show=False)
        
        
if steps.get("photometry", False):
    print("→ Photometry on science images")
    for objname in objects:
        print(f"   → Object: {objname}")
        print(f"      → Generating reference catalog")

        ra, dec = load_target_coordinates(objname, objects_csv)
        phot_cat_cfg = cfg["photometry"]["catalog"]

        ds = pd.read_csv(Path(night_dir, output_file))
        img_files = ds[(ds["OBJECT"]==objname)&(ds["ASTROMET"]=="yes")]["FILENAME"].values
        img_path = night_dir / img_files[0]
        phot_cat_path = generate_refcat(
            objname=objname, ra_center=ra, dec_center=dec,
            img_path=img_path, objects_dir=object_path,
            fov_frac=phot_cat_cfg.get("fov_frac", 0.3),
            min_mag=phot_cat_cfg.get("min_mag", 9),
            max_mag=phot_cat_cfg.get("max_mag", 12),
            max_mag_err=phot_cat_cfg.get("max_mag_err", 0.25),
            use_catalogs="all",
            plot=phot_cat_cfg.get("plot", False),
            type="phot",
            overwrite=phot_cat_cfg.get("overwrite", False)
        )
        print(f"      → Plotting catalog on combined images")
        combined_im = [f for f in dataset["images_combined"] if objname in f.name][0]
        plot_catalog_on_image(fits_file=combined_im,
                        catalog=phot_cat_path,
                        obj_ra=ra,
                        obj_dec=dec,
                        out_png=Path(object_path, objname, f"{objname}_phot_cat.png"),
                        title=f"{objname} – Aligment catalog")


        


print("✓ Pipeline finished successfully")

Running pipeline in night dir: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/20250823bis


New pipeline run: 2026-01-21 15:09:54
→ Updating FITS headers
→ Exporting metadata to images_data.csv


   Processing FITS files: 100%|██████████| 498/498 [00:00<00:00, 265340.88it/s]

→ Photometry on science images
   → Object: OGLE-2025-BLG-0451
      → Generating reference catalog
         Sobrescribiendo archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.csv
         FOV diagonal: 13.5' → Search radius: 4.0'
            • Catalog Gaia3 → 24 refs
      ✓ Archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.csv generado correctamente.
      → Plotting catalog on combined images


      ✓ Plot saved: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0451/OGLE-2025-BLG-0451_phot_cat.png
   → Object: OGLE-2025-BLG-0675
      → Generating reference catalog
         Sobrescribiendo archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0675/OGLE-2025-BLG-0675_phot_cat.csv
         FOV diagonal: 13.5' → Search radius: 4.0'
            • Catalog Gaia3 → 58 refs
      ✓ Archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0675/OGLE-2025-BLG-0675_phot_cat.csv generado correctamente.
      → Plotting catalog on combined images
      ✓ Plot saved: /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hsh/data/objetos/OGLE-2025-BLG-0675/OGLE-2025-BLG-0675_phot_cat.png
   → Object: OGLE-2025-GD-0001
      → Generating reference catalog
         Sobrescribiendo archivo /home/knowogrodski/Documentos/observation_HSH/ulens_hsh/ulens_hs

In [2]:
phot_cfg

{'use_catalogs': ['Gaia'],
 'query_method': 'gaia',
 'gaia_to_vi': 'ESA',
 'min_mag': 9,
 'max_mag': 15,
 'max_mag_err': 0.25,
 'fov_frac': 0.3,
 'overwrite': True,
 'min_bp_snr': 50,
 'min_rp_snr': 50,
 'min_g_snr': 100,
 'row_limit': -1}

In [ ]:
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
from astropy.table import Table

tap = TapPlus(url="http://tapvizier.u-strasbg.fr/TAPVizieR/tap")

# Query síncrona simple
query = """
SELECT TOP 10000 "II/246/out".RAJ2000, "II/246/out".DEJ2000, "II/246/out"."2MASS", "II/246/out".Jmag
FROM "II/246/out"
WHERE "II/246/out".Jmag < 15
"""

#job = tap.launch_job(query)           # síncrono por defecto

job = tap.launch_job_async(query)


table = job.get_results()

Created TAP+ (v20200428.1) - Connection:
	Host: tapvizier.u-strasbg.fr
	Use HTTPS: False
	Port: 80
	SSL Port: 443


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]


In [17]:
from astroquery.vizier import Vizier
from astropy.coordinates import SkyCoord
import astropy.units as u

v = Vizier(columns=["RA_ICRS", "DE_ICRS"],
             row_limit=10000)
coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
radius = 10 * u.arcmin
r = v.query_region(coord, radius=radius, catalog="J/ApJ/867/105/refcat2")

In [18]:
r

TableList with 1 tables:
	'0:J/ApJ/867/105/refcat2' with 2 column(s) and 10000 row(s) 